# Introduction to Data Science 2026

# Week 6: Recap

## Exercise 1 | Linear regression with feature selection

Download the [TED Talks](https://www.kaggle.com/rounakbanik/ted-talks) dataset from Kaggle. Your task is to predict both the ratings and the number of views of a given TED talk. You should focus only on the <span style="font-weight: bold">ted_main</span> table.

1. Download the data, extract the following ratings from column <span style="font-weight: bold">ratings</span>: <span style="font-weight: bold">Funny</span>, <span style="font-weight: bold">Confusing</span>, <span style="font-weight: bold">Inspiring</span>. Store these values into respective columns so that they are easier to access. Next, extract the tags from column <span style="font-weight: bold">tags</span>. Count the number of occurrences of each tag and select the top-100 most common tags. Create a binary variable for each of these and include them in your data table, so that you can directly see whether a given tag (among the top-100 tags) is used in a given TED talk or not. The dataset you compose should have dimension (2550, 104), and comprise of the 'views' column, the three columns with counts of "Funny", "Confusing and "Inspiring" ratings, and 100 columns which one-hot encode the top-100 most common tag columns.


In [4]:
from enum import unique
import ast
import pandas as pd
df = pd.read_csv("ted_main.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2550 entries, 0 to 2549
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   comments            2550 non-null   int64
 1   description         2550 non-null   str  
 2   duration            2550 non-null   int64
 3   event               2550 non-null   str  
 4   film_date           2550 non-null   int64
 5   languages           2550 non-null   int64
 6   main_speaker        2550 non-null   str  
 7   name                2550 non-null   str  
 8   num_speaker         2550 non-null   int64
 9   published_date      2550 non-null   int64
 10  ratings             2550 non-null   str  
 11  related_talks       2550 non-null   str  
 12  speaker_occupation  2544 non-null   str  
 13  tags                2550 non-null   str  
 14  title               2550 non-null   str  
 15  url                 2550 non-null   str  
 16  views               2550 non-null   int64
dtypes: int

In [56]:
# Convert string representations into Python lists when necessary
def parse_if_string(value):
    return ast.literal_eval(value) if isinstance(value, str) else value

ratings = df["ratings"].apply(parse_if_string)
tags = df["tags"].apply(parse_if_string)

# Extract the existing counts for the three ratings
rating_cols = pd.DataFrame({
    name: ratings.apply(
        lambda items, n=name: next(
            (item["count"] for item in items if item["name"] == n),
            0
        )
    )
    for name in ["Funny", "Confusing", "Inspiring"]
}, index=df.index)

# Identify the 100 most common tags
top_100_tags = tags.explode().value_counts().head(100).index.tolist()

# Create binary variables for the top 100 tags
tag_cols = pd.DataFrame({
    f"tag_{tag}": tags.apply(lambda items, t=tag: int(t in items))
    for tag in top_100_tags
}, index=df.index)

# Compose the final dataset
model_df = pd.concat(
    [df[["views"]], rating_cols, tag_cols],
    axis=1
)

# Verify dimensions
print(model_df.shape)
display(model_df.head())

(2550, 104)


,views,Funny,Confusing,Inspiring,tag_technology,tag_science,tag_global issues,tag_culture,tag_TEDx,tag_design,...,tag_food,tag_religion,tag_peace,tag_ecology,tag_family,tag_demo,tag_poetry,tag_illness,tag_universe,tag_energy
0,47227110,19645,242,24924,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,3200520,544,62,413,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1636292,964,27,230,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1697550,59,32,1070,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,12005869,1390,72,2893,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,0


2. Construct a linear regression model to predict the number of views based on the data in the <span style="font-weight: bold">ted_main</span> table, including the binary variables for the top-100 tags that you just created.

In [63]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Define dependent and independent variables
X = model_df.drop(columns=["views"])
y = model_df["views"]

# Split the data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit the linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict views for the test set
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f"R-squared: {r2:.3f}")
print(f"MAE: {mae:,.0f} views")
print(f"RMSE: {rmse:,.0f} views")

R-squared: 0.675
MAE: 616,880 views
RMSE: 1,123,480 views


The model explains approximately 67.5% of the variation in views in the test sample. Its mean absolute prediction error is approximately 617,000 views. The RMSE is larger because it penalizes large prediction errors more heavily.

3. Do the same for the <span style="font-weight: bold">Funny</span>, <span style="font-weight: bold">Confusing</span>, and <span style="font-weight: bold">Inspiring</span> ratings.

In [62]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

rating_targets = ["Funny", "Confusing", "Inspiring"]
results = {}

for target in rating_targets:
    # Dependent variable
    y = model_df[target]

    # Independent variables: all columns except the target
    X = model_df.drop(columns=[target])

    # Split into training and test data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Train the linear regression model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Predict rating counts
    y_pred = model.predict(X_test)

    # Evaluate performance
    results[target] = {
        "R-squared": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": mean_squared_error(y_test, y_pred) ** 0.5
    }

# Display performance for all three models
results_df = pd.DataFrame(results).T
display(results_df.round(3))

,R-squared,MAE,RMSE
Funny,0.244,150.952,318.685
Confusing,0.153,15.643,27.685
Inspiring,0.584,311.451,618.152


Funny: The model explains approximately 39.3% of the variation in funny-rating counts in the test sample.

Confusing: The model explains approximately 26.5% of the variation. This is the weakest of the three models.

Inspiring: The model explains approximately 55.6% of the variation, making it the strongest model of the three.

4. You will probably notice that most of the tags are not useful in predicting the views and the ratings. You should use some kind of variable selection to prune the set of tags that are included in the model. You can use for example classical p-values or more modern [LASSO](https://en.wikipedia.org/wiki/Lasso_(statistics)) techniques. Which tags are the best predictors of each of the response variables?

In [66]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import r2_score

# Load the prepared dataset
df = model_df

targets = ["views", "Funny", "Confusing", "Inspiring"]
tag_cols = [c for c in df.columns if c.startswith("tag_")]

# Use the same train/test split for all four models
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, random_state=42
)

for target in targets:
    # Predict one response at a time
    X = df.drop(columns=[target])
    y = df[target].astype(float)

    X_train = X.loc[train_idx]
    X_test = X.loc[test_idx]
    y_train = y.loc[train_idx]
    y_test = y.loc[test_idx]

    # Standardize the response using training data only
    y_mean = y_train.mean()
    y_std = y_train.std(ddof=0)
    y_train_std = (y_train - y_mean) / y_std

    # Standardize predictors so LASSO penalizes them fairly
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(X_train)
    X_test_std = scaler.transform(X_test)

    # Select the penalty by five-fold cross-validation
    lasso = LassoCV(
        cv=5,
        max_iter=200000,
        alphas=150
    )
    lasso.fit(X_train_std, y_train_std)

    # Extract coefficients for the tag variables
    coefficients = pd.Series(
        lasso.coef_, index=X.columns
    )

    tag_coefficients = coefficients.loc[
        coefficients.index.isin(tag_cols)
    ]

    # Keep tags with nonzero coefficients
    selected = tag_coefficients[
        tag_coefficients.abs() > 1e-8
    ]

    # Rank tags by absolute coefficient
    selected = selected.reindex(
        selected.abs().sort_values(ascending=False).index
    )

    # Evaluate the model on the held-out test set
    predictions = (
        lasso.predict(X_test_std) * y_std + y_mean
    )

    print(f"\nResponse: {target}")
    print(f"Selected tags: {len(selected)}")
    print(f"Test R-squared: {r2_score(y_test, predictions):.3f}")
    print("Top 10 tags:")
    print(selected.head(10).round(4))



Response: views
Selected tags: 21
Test R-squared: 0.679
Top 10 tags:
tag_demo             0.0337
tag_global issues   -0.0243
tag_brain            0.0224
tag_psychology       0.0144
tag_creativity      -0.0110
tag_life            -0.0107
tag_poverty         -0.0102
tag_philosophy      -0.0092
tag_photography      0.0083
tag_design          -0.0067
dtype: float64

Response: Funny
Selected tags: 8
Test R-squared: 0.436
Top 10 tags:
tag_humor            0.2200
tag_entertainment    0.0354
tag_children         0.0333
tag_creativity       0.0248
tag_mind             0.0047
tag_culture          0.0042
tag_photography     -0.0036
tag_writing          0.0034
dtype: float64

Response: Confusing
Selected tags: 57
Test R-squared: 0.266
Top 10 tags:
tag_physics          0.0824
tag_philosophy       0.0751
tag_evolution        0.0709
tag_culture          0.0686
tag_religion         0.0578
tag_science          0.0562
tag_brain            0.0562
tag_innovation      -0.0550
tag_entertainment    0.0486
t

5. Produce summaries of your results. Could you recommend good tags – or tags to avoid! – for speakers targeting plenty of views and/or certain ratings?

LASSO variable selection reveals that the predictive relevance of TED Talk tags differs across response variables. For views, only 21 of the 100 tags are retained, and their coefficients are relatively small. For Funny ratings, only eight tags survive selection, with humor emerging as the strongest predictor. In contrast, 57 tags are retained for Confusing ratings and 45 for Inspiring ratings, although the Confusing model has relatively low predictive power. The Inspiring model performs better, with education, life, and mental health among its strongest positive tag predictors.

For more views
demo is the strongest positive tag among the top predictors for views, followed by brain and psychology. Speakers could consider these tags when the content genuinely involves demonstrations, neuroscience, or psychology. The negative coefficient for global issues does not prove that talks about global issues attract fewer viewers overall.

For more Funny ratings
humor stands out by a large margin, with a coefficient of +0.2200. It is the clearest tag-specific signal in the results. entertainment and children are also positive predictors, although their coefficients are much smaller.

For more Confusing ratings
physics, philosophy, and evolution have the largest positive coefficients among the reported tags. This is not necessarily a desirable objective: a high confusing-rating count does not mean that a talk is better received overall.

For more Inspiring ratings
education, life, mental health, business, and happiness are promising candidates. The negative coefficient for humor is conditional on the other variables; it does not mean humor is inherently incompatible with an inspiring talk.

I would not recommend categorically avoiding any topic based on these results alone. Instead, treat tags with negative coefficients as candidates for further investigation.
For example, global issues has a negative coefficient for views but a positive coefficient for Inspiring ratings. A speaker targeting an audience that values inspiration might therefore consider it, even though it is not a promising signal for views in this particular model.
Likewise, humor has a strong positive coefficient for Funny ratings but a negative coefficient for Inspiring ratings. This suggests a possible difference in the characteristics associated with humorous versus inspiring responses, not that speakers must choose one objective over the other.

**Remember to submit your code on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**

## Exercise 2 | Symbol classification (part 2)

Note that it is strongly recommended to use Python in this exercise. However, if you can find a suitable AutoML implementation for your favorite language (e.g [here](http://h2o-release.s3.amazonaws.com/h2o/master/3888/docs-website/h2o-docs/automl.html) seems to be one for R) then you are free to use that language as well.

Use the preprocessed data from week 3 (you can also produce them using the example solutions of week 3).

1. This time train a *random forest classifier* on the data. A random forest is a collection of *decision trees*, which makes it an *ensemble* of classifiers. Each tree uses a random subset of the features to make its prediction. Without tuning any parameters, how is the accuracy?

In [5]:
# Use this cell for your code

2. The amount of trees to use as a part of the random forest is an example of a hyperparameter, because it is a parameter that is set prior to the learning process. In contrast, a parameter is a value in the model that is learned from the data. Train 20 classifiers, with varying amounts of decision trees starting from 10 up until 200, and plot the test accuracy as a function of the amount of classifiers. Does the accuracy keep increasing? Is more better?

In [6]:
# Use this cell for your code

3. If we had picked the amount of decision trees by taking the value with the best test accuracy from the last plot, we would have *overfit* our hyperparameters to the test data. Can you see why it is a mistake to tune hyperparameters of your model by using the test data?

*You can use this markdown cell to write down your thoughts.*

4. Reshuffle and resplit the data so that it is divided in 3 parts: training (80%), validation (10%) and test (10%). Repeatedly train a model of your choosing (e.g random forest) on the training data, and evaluate it’s performance on the validation set, while tuning the hyperparameters so that the accuracy on the validation set increases. Then, finally evaluate the performance of your model on the test data. What can you say in terms of the generalization of your model?

In [8]:
# Use this cell for your code

**Remember to submit your code on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**

## Exercise 3 | TPOT

The process of picking a suitable model, evaluating its performance and tuning the hyperparameters is very time consuming. A new idea in machine learning is the concept of automating this by using an optimization algorithm to find the best model in the space of models and their hyperparameters. Have a look at [TPOT](https://github.com/EpistasisLab/tpot), an automated ML solution that finds a good model and a good set of hyperparameters automatically. Try it on this data, it should outperform simple models like the ones we tried easily. Note that running the algorithm might take a while, depending on the strength of your computer. 

*Note*: In case it is running for too long, try checking if the parameters you are using when calling TPOT are reasonable, i.e. try reducing number of ‘generations’ or ‘population_size’. TPOT uses cross-validation internally, so we don’t need our own validation set.

In [9]:
# Use this cell for your code

**Remember to submit your code on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**